# 第二章：循環神經網路的不合理有效性 - PyTorch 實作
## The Unreasonable Effectiveness of Recurrent Neural Networks

**原文：Andrej Karpathy (2015)**  
**連結：http://karpathy.github.io/2015/05/21/rnn-effectiveness/**

---

### 本章重點

1. **字元級語言模型**：一個字元一個字元地學習和生成文字
2. **RNN 的核心公式**：$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$
3. **反向傳播穿越時間 (BPTT)**：如何訓練循環網路
4. **溫度採樣**：控制生成文字的創造性

In [ ]:
# 跨平台中文字型設定（支援 Colab、VSCode、Antigravity 等本地環境）
import subprocess
import os
import shutil
import platform

system = platform.system()

# 必須在 import matplotlib 之前清除快取
cache_dir = os.path.expanduser('~/.matplotlib')
if os.path.exists(cache_dir):
    for f in os.listdir(cache_dir):
        if f.startswith('fontlist'):
            try:
                os.remove(os.path.join(cache_dir, f))
            except:
                pass

cache_dir2 = os.path.expanduser('~/.cache/matplotlib')
if os.path.exists(cache_dir2):
    shutil.rmtree(cache_dir2, ignore_errors=True)

# Linux/Colab 環境安裝中文字型
if system == 'Linux' or 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    try:
        result = subprocess.run(['fc-list', ':lang=zh'], capture_output=True, text=True)
        if 'Noto Sans CJK' not in result.stdout:
            print("正在安裝中文字型...")
            subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
            subprocess.run(['apt-get', 'install', '-qq', '-y', 'fonts-noto-cjk'], capture_output=True)
            print("中文字型安裝完成，請重新啟動 kernel")
    except:
        pass

print(f"✓ {system} 環境")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# 重建字型快取並設定中文字型
from matplotlib.font_manager import fontManager
fontManager.__init__()

chinese_fonts = [
    'Heiti TC', 'PingFang TC', 'Noto Sans CJK TC',
    'Heiti SC', 'PingFang SC', 'Noto Sans CJK SC', 
    'Microsoft JhengHei', 'Microsoft YaHei',
    'SimHei', 'WenQuanYi Micro Hei', 'Arial Unicode MS',
]

available_fonts = set(f.name for f in fontManager.ttflist)
selected_font = None
for font in chinese_fonts:
    if font in available_fonts:
        selected_font = font
        break

if selected_font:
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [selected_font] + chinese_fonts
    plt.rcParams['axes.unicode_minus'] = False
    print(f"✓ 使用中文字型: {selected_font}")
else:
    plt.rcParams['font.sans-serif'] = chinese_fonts
    plt.rcParams['axes.unicode_minus'] = False
    print("⚠ 使用預設字型列表")

import numpy as np
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 設定隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 跨平台中文字型設定
# 字型優先順序：涵蓋 Mac, Windows, Linux, Colab 常見中文字型
chinese_fonts = [
    'Noto Sans CJK TC',      # Linux/Colab/Mac(安裝後)
    'Noto Sans CJK SC',      # Linux/Colab/Mac(安裝後) 簡體
    'Heiti TC',              # Mac 內建 繁體
    'Heiti SC',              # Mac 內建 簡體
    'PingFang TC',           # Mac 內建 繁體
    'PingFang SC',           # Mac 內建 簡體
    'Microsoft JhengHei',    # Windows 微軟正黑體 繁體
    'Microsoft YaHei',       # Windows 微軟雅黑 簡體
    'SimHei',                # Windows 黑體
    'WenQuanYi Micro Hei',   # Linux 文泉驛微米黑
    'Droid Sans Fallback',   # Android/舊版 Linux
    'Arial Unicode MS',      # 跨平台 Unicode 字型
    'DejaVu Sans',           # 後備字型
]

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = chinese_fonts
plt.rcParams['axes.unicode_minus'] = False  # 正確顯示負號

# 驗證字型設定
def verify_chinese_font():
    """驗證中文字型是否正確載入"""
    from matplotlib.font_manager import FontManager
    fm = FontManager()
    available = set(f.name for f in fm.ttflist)
    for font in chinese_fonts:
        if font in available:
            print(f"✓ 使用中文字型: {font}")
            return font
    print("⚠ 警告：未找到中文字型，中文可能無法正確顯示")
    return None

# 設定設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用設備: {device}")
verify_chinese_font()

---

## 1. 準備訓練資料

我們使用簡單的合成文字來訓練。在實際應用中，你可以使用莎士比亞、程式碼、或任何文字。

In [ ]:
# 訓練資料：簡單的句子模式
data = """
hello world
hello deep learning
deep neural networks
neural networks learn patterns
patterns in data
data drives learning
learning from examples
examples help networks
networks process information
information is everywhere
everywhere you look data
look at neural networks
neural learning is powerful
powerful patterns emerge
emerge from deep networks
deep learning revolution
revolution in artificial intelligence
intelligence from data
""" * 20  # 重複以獲得更多訓練資料

print(f"資料長度: {len(data)} 字元")
print(f"前 200 個字元:")
print(repr(data[:200]))

In [ ]:
# 建立詞彙表
chars = sorted(list(set(data)))
vocab_size = len(chars)

# 字元到索引的映射
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"詞彙表大小: {vocab_size}")
print(f"詞彙表: {repr(''.join(chars))}")
print(f"\n字元到索引映射範例:")
for ch in ['a', 'e', 'i', 'o', 'u', ' ', '\n']:
    if ch in char_to_idx:
        print(f"  '{repr(ch)}' -> {char_to_idx[ch]}")

In [ ]:
# 將整個資料轉換成索引
data_indices = torch.tensor([char_to_idx[ch] for ch in data], dtype=torch.long)
print(f"資料張量形狀: {data_indices.shape}")
print(f"前 50 個索引: {data_indices[:50].tolist()}")

# 顯示對應的字元
print(f"對應字元: {repr(''.join([idx_to_char[i.item()] for i in data_indices[:50]]))}")

---

## 2. 從零實作 Vanilla RNN

### 2.1 RNN 的核心公式

**隱藏狀態更新：**
$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

**輸出計算：**
$$y_t = W_{hy} \cdot h_t + b_y$$

**機率分佈：**
$$p_t = \text{softmax}(y_t)$$

In [ ]:
class VanillaRNN(nn.Module):
    """
    從零實作的 Vanilla RNN
    
    這個實作故意不使用 nn.RNN，而是手動實現所有計算，
    以便更好地理解 RNN 的工作原理。
    """
    
    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        
        # 權重矩陣
        # W_xh: 輸入到隱藏層 (hidden_size x vocab_size)
        self.W_xh = nn.Parameter(torch.randn(hidden_size, vocab_size) * 0.01)
        
        # W_hh: 隱藏層到隱藏層（這是「循環」的核心！）
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        
        # W_hy: 隱藏層到輸出
        self.W_hy = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.01)
        
        # 偏置
        self.b_h = nn.Parameter(torch.zeros(hidden_size, 1))
        self.b_y = nn.Parameter(torch.zeros(vocab_size, 1))
    
    def forward(self, inputs: torch.Tensor, h_prev: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor], torch.Tensor]:
        """
        前向傳播
        
        Args:
            inputs: 輸入序列的索引 (seq_len,)
            h_prev: 初始隱藏狀態 (hidden_size, 1)
        
        Returns:
            outputs: 每個時間步的 logits (seq_len, vocab_size)
            hidden_states: 每個時間步的隱藏狀態
            h_last: 最後一個隱藏狀態
        """
        outputs = []
        hidden_states = []
        h = h_prev
        
        for t in range(len(inputs)):
            # One-hot 編碼當前輸入
            x = torch.zeros(self.vocab_size, 1, device=inputs.device)
            x[inputs[t]] = 1.0
            
            # 核心公式: h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
            h = torch.tanh(
                torch.mm(self.W_xh, x) +      # 當前輸入的影響
                torch.mm(self.W_hh, h) +      # 歷史記憶的影響
                self.b_h
            )
            hidden_states.append(h)
            
            # 輸出: y_t = W_hy * h_t + b_y
            y = torch.mm(self.W_hy, h) + self.b_y
            outputs.append(y.squeeze())
        
        return torch.stack(outputs), hidden_states, h
    
    def init_hidden(self, device=None) -> torch.Tensor:
        """初始化隱藏狀態為零向量"""
        if device is None:
            device = next(self.parameters()).device
        return torch.zeros(self.hidden_size, 1, device=device)
    
    def sample(self, h: torch.Tensor, seed_idx: int, n: int, 
               temperature: float = 1.0) -> List[int]:
        """
        從模型中採樣生成文字
        
        Args:
            h: 初始隱藏狀態
            seed_idx: 種子字元的索引
            n: 要生成的字元數量
            temperature: 溫度參數（越高越隨機）
        
        Returns:
            生成的字元索引列表
        """
        indices = []
        x = torch.zeros(self.vocab_size, 1, device=h.device)
        x[seed_idx] = 1.0
        
        with torch.no_grad():
            for _ in range(n):
                # 前向傳播一步
                h = torch.tanh(
                    torch.mm(self.W_xh, x) +
                    torch.mm(self.W_hh, h) +
                    self.b_h
                )
                y = torch.mm(self.W_hy, h) + self.b_y
                
                # 應用溫度並計算機率
                probs = F.softmax(y.squeeze() / temperature, dim=0)
                
                # 根據機率採樣
                idx = torch.multinomial(probs, 1).item()
                indices.append(idx)
                
                # 準備下一步的輸入
                x = torch.zeros(self.vocab_size, 1, device=h.device)
                x[idx] = 1.0
        
        return indices


# 建立模型
hidden_size = 128
model = VanillaRNN(vocab_size, hidden_size).to(device)

print(f"模型架構:")
print(f"  詞彙表大小: {vocab_size}")
print(f"  隱藏層大小: {hidden_size}")
print(f"\n參數數量:")
total_params = 0
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape} = {param.numel()} 參數")
    total_params += param.numel()
print(f"  總計: {total_params} 參數")

### 2.2 理解 RNN 的「循環」

讓我們視覺化 RNN 如何處理一個短序列：

In [ ]:
# 用一個簡單的例子展示 RNN 的工作流程
test_text = "hello"
test_indices = torch.tensor([char_to_idx[ch] for ch in test_text], device=device)

print("RNN 處理 'hello' 的過程：")
print("=" * 50)

h = model.init_hidden(device)
print(f"初始隱藏狀態 h_0: 形狀 {h.shape}, 全為零\n")

for t, (ch, idx) in enumerate(zip(test_text, test_indices)):
    # 手動計算一步
    x = torch.zeros(vocab_size, 1, device=device)
    x[idx] = 1.0
    
    h_new = torch.tanh(
        torch.mm(model.W_xh, x) +
        torch.mm(model.W_hh, h) +
        model.b_h
    )
    
    y = torch.mm(model.W_hy, h_new) + model.b_y
    probs = F.softmax(y.squeeze(), dim=0)
    
    # 找出最可能的下一個字元
    top_prob, top_idx = probs.topk(3)
    top_chars = [idx_to_char[i.item()] for i in top_idx]
    
    print(f"時間步 t={t}: 輸入 '{ch}'")
    print(f"  隱藏狀態變化: h_{t} -> h_{t+1}")
    print(f"  前 3 個預測: {[(c, f'{p:.3f}') for c, p in zip(top_chars, top_prob.tolist())]}")
    print()
    
    h = h_new

---

## 3. 訓練迴圈

### 3.1 準備訓練批次

In [ ]:
def get_batch(data: torch.Tensor, seq_length: int, batch_start: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    獲取一個訓練批次
    
    Args:
        data: 整個資料的索引張量
        seq_length: 序列長度
        batch_start: 批次在資料中的起始位置
    
    Returns:
        inputs: 輸入序列 (seq_length,)
        targets: 目標序列 (seq_length,) - 比輸入右移一位
    """
    inputs = data[batch_start:batch_start + seq_length]
    targets = data[batch_start + 1:batch_start + seq_length + 1]
    return inputs, targets


# 測試批次生成
seq_length = 25
inputs, targets = get_batch(data_indices, seq_length, 0)

print(f"序列長度: {seq_length}")
print(f"\n輸入序列:")
print(f"  索引: {inputs.tolist()}")
print(f"  字元: '{repr(''.join([idx_to_char[i.item()] for i in inputs]))}'")
print(f"\n目標序列:")
print(f"  索引: {targets.tolist()}")
print(f"  字元: '{repr(''.join([idx_to_char[i.item()] for i in targets]))}'")
print(f"\n注意: 目標是輸入右移一位（預測下一個字元）")

### 3.2 訓練函數

In [ ]:
def train_rnn(model: VanillaRNN, 
              data: torch.Tensor,
              num_iterations: int = 3000,
              seq_length: int = 25,
              learning_rate: float = 0.01,
              print_every: int = 300) -> List[float]:
    """
    訓練 RNN
    
    Args:
        model: RNN 模型
        data: 訓練資料（索引張量）
        num_iterations: 訓練迭代次數
        seq_length: 序列長度
        learning_rate: 學習率
        print_every: 每隔多少步印出進度
    
    Returns:
        losses: 損失歷史
    """
    # 使用 Adam 優化器（比 Adagrad 更現代）
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 損失函數
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    smooth_loss = -np.log(1.0 / vocab_size) * seq_length  # 初始損失估計
    
    # 資料指針
    data_ptr = 0
    
    # 初始隱藏狀態
    h = model.init_hidden(device)
    
    for iteration in range(num_iterations):
        # 檢查是否需要重置
        if data_ptr + seq_length + 1 >= len(data):
            data_ptr = 0
            h = model.init_hidden(device)
        
        # 獲取批次
        inputs, targets = get_batch(data, seq_length, data_ptr)
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        # 分離隱藏狀態（截斷 BPTT）
        h = h.detach()
        
        # 前向傳播
        outputs, _, h = model(inputs, h)
        
        # 計算損失
        loss = criterion(outputs, targets)
        
        # 反向傳播
        optimizer.zero_grad()
        loss.backward()
        
        # 梯度裁剪（防止梯度爆炸）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        # 更新參數
        optimizer.step()
        
        # 記錄損失
        smooth_loss = smooth_loss * 0.999 + loss.item() * 0.001
        losses.append(smooth_loss)
        
        # 印出進度和樣本
        if iteration % print_every == 0:
            print(f"\n{'='*60}")
            print(f"迭代 {iteration}, 損失: {smooth_loss:.4f}")
            print(f"{'='*60}")
            
            # 生成樣本
            sample_h = model.init_hidden(device)
            seed_idx = inputs[0].item()
            sample_indices = model.sample(sample_h, seed_idx, 100, temperature=0.8)
            sample_text = ''.join([idx_to_char[i] for i in sample_indices])
            print(f"生成樣本 (種子: '{idx_to_char[seed_idx]}'):")
            print(sample_text)
        
        # 移動資料指針
        data_ptr += seq_length
    
    return losses

### 3.3 開始訓練

In [ ]:
# 訓練模型
print("開始訓練 RNN...")
print(f"設備: {device}")
print(f"隱藏層大小: {hidden_size}")
print(f"序列長度: 25")
print(f"迭代次數: 3000")

losses = train_rnn(
    model, 
    data_indices,
    num_iterations=3000,
    seq_length=25,
    learning_rate=0.01,
    print_every=500
)

---

## 4. 視覺化訓練過程

In [ ]:
# 繪製損失曲線
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(losses, linewidth=1, alpha=0.8)
plt.xlabel('迭代次數')
plt.ylabel('平滑損失')
plt.title('RNN 訓練損失曲線')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(losses[100:], linewidth=1, alpha=0.8)  # 跳過初始不穩定階段
plt.xlabel('迭代次數 (從 100 開始)')
plt.ylabel('平滑損失')
plt.title('RNN 訓練損失曲線 (放大)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n訓練統計:")
print(f"  初始損失: {losses[0]:.4f}")
print(f"  最終損失: {losses[-1]:.4f}")
print(f"  最低損失: {min(losses):.4f}")

---

## 5. 生成文字

### 5.1 不同溫度的效果

In [ ]:
# 用不同溫度生成文字
temperatures = [0.5, 0.8, 1.0, 1.2, 1.5]

print("溫度參數對生成文字的影響")
print("=" * 60)
print("\n溫度越低 → 越保守（選擇最可能的字元）")
print("溫度越高 → 越隨機（更多創意，但可能出錯）\n")

seed_char = 'd'  # 使用 'd' 作為種子（常見的開頭）
seed_idx = char_to_idx[seed_char]

for temp in temperatures:
    h = model.init_hidden(device)
    sample_indices = model.sample(h, seed_idx, 100, temperature=temp)
    sample_text = ''.join([idx_to_char[i] for i in sample_indices])
    
    print(f"溫度 = {temp}:")
    print(f"{seed_char}{sample_text}")
    print()

### 5.2 從不同種子字元生成

In [ ]:
# 從不同種子生成
seed_chars = ['h', 'n', 'l', 'd', 'p']

print("從不同種子字元生成文字 (溫度=0.8)")
print("=" * 60)

for seed_char in seed_chars:
    if seed_char in char_to_idx:
        seed_idx = char_to_idx[seed_char]
        h = model.init_hidden(device)
        sample_indices = model.sample(h, seed_idx, 80, temperature=0.8)
        sample_text = ''.join([idx_to_char[i] for i in sample_indices])
        
        print(f"種子 '{seed_char}':")
        print(f"  {seed_char}{sample_text}")
        print()

---

## 6. 視覺化隱藏狀態

觀察 RNN 的「記憶」如何隨著輸入變化。

In [ ]:
# 選擇一段文字來分析
analyze_text = "hello deep learning"
analyze_indices = torch.tensor([char_to_idx[ch] for ch in analyze_text], device=device)

# 前向傳播並收集隱藏狀態
h = model.init_hidden(device)
outputs, hidden_states, _ = model(analyze_indices, h)

# 將隱藏狀態轉換成矩陣
hidden_matrix = torch.stack([h.squeeze() for h in hidden_states]).detach().cpu().numpy()

print(f"分析文字: '{analyze_text}'")
print(f"隱藏狀態矩陣形狀: {hidden_matrix.shape} (時間步 x 隱藏維度)")

In [ ]:
# 視覺化隱藏狀態
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# 上圖：完整的隱藏狀態熱圖
im1 = axes[0].imshow(hidden_matrix.T, cmap='RdBu', aspect='auto', 
                      interpolation='nearest', vmin=-1, vmax=1)
axes[0].set_xlabel('時間步（字元位置）')
axes[0].set_ylabel('隱藏單元')
axes[0].set_title(f"RNN 隱藏狀態演化 - '{analyze_text}'")
axes[0].set_xticks(range(len(analyze_text)))
axes[0].set_xticklabels(list(analyze_text))
plt.colorbar(im1, ax=axes[0], label='活化值')

# 下圖：選擇幾個有趣的隱藏單元
# 找出變化最大的幾個單元
variance = hidden_matrix.var(axis=0)
top_units = variance.argsort()[-5:][::-1]  # 前 5 個變化最大的

for i, unit in enumerate(top_units):
    axes[1].plot(hidden_matrix[:, unit], label=f'單元 {unit}', linewidth=2, marker='o')

axes[1].set_xlabel('時間步（字元位置）')
axes[1].set_ylabel('活化值')
axes[1].set_title('變化最大的 5 個隱藏單元')
axes[1].set_xticks(range(len(analyze_text)))
axes[1].set_xticklabels(list(analyze_text))
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n觀察:")
print("• 某些隱藏單元對特定字元（如空格）反應強烈")
print("• 隱藏狀態在詞邊界處有明顯變化")
print("• 這展示了 RNN 如何『記住』和『編碼』序列資訊")

---

## 7. 預測機率分佈視覺化

In [ ]:
# 視覺化預測機率
probs = F.softmax(outputs, dim=1).detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：預測機率熱圖
im = axes[0].imshow(probs.T, cmap='YlOrRd', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('時間步（輸入字元）')
axes[0].set_ylabel('詞彙表索引')
axes[0].set_title('每個時間步的預測機率分佈')
axes[0].set_xticks(range(len(analyze_text)))
axes[0].set_xticklabels(list(analyze_text))
plt.colorbar(im, ax=axes[0], label='機率')

# 右圖：選擇幾個時間步展示詳細分佈
selected_steps = [0, 5, 10, 15]  # 選擇幾個代表性時間步
x = np.arange(vocab_size)
width = 0.2

for i, step in enumerate(selected_steps):
    if step < len(analyze_text):
        axes[1].bar(x + i * width, probs[step], width, 
                    label=f"t={step} ('{analyze_text[step]}')")

axes[1].set_xlabel('字元索引')
axes[1].set_ylabel('機率')
axes[1].set_title('選定時間步的預測機率分佈')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 8. 與 PyTorch 內建 RNN 比較

讓我們也用 PyTorch 的內建 `nn.RNN` 實作一個版本來比較。

In [ ]:
class CharRNNBuiltin(nn.Module):
    """
    使用 PyTorch 內建 RNN 的版本
    """
    def __init__(self, vocab_size: int, hidden_size: int, num_layers: int = 1):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Embedding 層（比 one-hot 更高效）
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        
        # RNN 層
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)
        
        # 輸出層
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, hidden_size)
        output, h = self.rnn(embedded, h)  # output: (batch, seq_len, hidden_size)
        output = self.fc(output)  # (batch, seq_len, vocab_size)
        return output, h
    
    def init_hidden(self, batch_size=1):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)


# 建立內建版本
model_builtin = CharRNNBuiltin(vocab_size, hidden_size).to(device)

print("PyTorch 內建 RNN 版本:")
print(model_builtin)
print(f"\n參數數量: {sum(p.numel() for p in model_builtin.parameters())}")

---

## 9. 關鍵洞見總結

In [ ]:
# 創建總結視覺化
fig = plt.figure(figsize=(16, 10))

# 1. RNN 架構圖（文字描述）
ax1 = fig.add_subplot(2, 2, 1)
ax1.axis('off')
architecture_text = """
RNN 核心架構

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    x_t ──────────────────────┐
         ↓                    │
    ┌─────────────────────┐   │
    │  h_t = tanh(         │   │
    │    W_xh·x_t +        │←──┘
    │    W_hh·h_{t-1} +    │
    │    b_h               │
    │  )                   │
    └─────────────────────┘
         ↓         ↑
        y_t        │
                h_{t-1}
                (循環連接)

關鍵: W_hh 允許資訊跨時間步流動
"""
ax1.text(0.1, 0.5, architecture_text, fontsize=10,
         verticalalignment='center', transform=ax1.transAxes)
ax1.set_title('RNN 架構', fontsize=12, fontweight='bold')

# 2. 訓練損失
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(losses, linewidth=1, alpha=0.8)
ax2.set_xlabel('迭代次數')
ax2.set_ylabel('損失')
ax2.set_title('訓練損失曲線', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. 隱藏狀態
ax3 = fig.add_subplot(2, 2, 3)
ax3.imshow(hidden_matrix[:, :32].T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
ax3.set_xlabel('時間步')
ax3.set_ylabel('隱藏單元 (前32個)')
ax3.set_title(f"隱藏狀態演化 ('{analyze_text}')", fontsize=12, fontweight='bold')
ax3.set_xticks(range(len(analyze_text)))
ax3.set_xticklabels(list(analyze_text), fontsize=8)

# 4. 關鍵洞見
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')
insights_text = """
「不合理的有效性」關鍵洞見

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 字元級建模
   • 不需要分詞，不需要詞彙表
   • 可以處理任何語言和格式
   • 學會拼寫、標點、格式

2. 隱藏狀態 = 記憶
   • 攜帶所有歷史資訊的壓縮表示
   • 自動學習什麼值得記住

3. 端到端學習
   • 從原始字元到語言模型
   • 無需人工特徵工程

4. 通往 GPT 的道路
   • 自回歸語言建模
   • 預測下一個 token
   • Scale up = 更強能力
"""
ax4.text(0.05, 0.5, insights_text, fontsize=10,
         verticalalignment='center', transform=ax4.transAxes)
ax4.set_title('關鍵洞見', fontsize=12, fontweight='bold')

plt.suptitle('第二章總結：循環神經網路的不合理有效性', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 10. 思考題

1. **為什麼字元級模型比詞級模型更能處理未見過的詞？**

2. **如果訓練資料中某個字元只出現過一次，模型能學會正確使用它嗎？**

3. **為什麼我們使用 tanh 而不是 ReLU 作為活化函數？**

4. **梯度裁剪是如何防止訓練不穩定的？**

5. **溫度參數是如何影響生成多樣性的？嘗試用數學解釋。**

---

## 參考資料

**原文連結：**
- 部落格：http://karpathy.github.io/2015/05/21/rnn-effectiveness/
- 作者：Andrej Karpathy

**相關程式碼：**
- min-char-rnn：https://gist.github.com/karpathy/d4dee566867f8291f086

**延伸閱讀：**
- Hochreiter & Schmidhuber (1997). *Long Short-Term Memory*
- Graves (2013). *Generating Sequences With Recurrent Neural Networks*

---

*下一章：Understanding LSTM Networks — 深入理解長短期記憶網路*